In [63]:
import torch
from PIL import Image
import pandas as pd
from transformers import AutoProcessor, Blip2Processor, Blip2ForImageTextRetrieval,  Blip2ForConditionalGeneration, BitsAndBytesConfig
import os

from translate_dn_en import translate_danish_to_english

In [64]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [65]:
processor = Blip2Processor.from_pretrained("Salesforce/blip2-flan-t5-xl")
model = Blip2ForConditionalGeneration.from_pretrained("Salesforce/blip2-flan-t5-xl")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [66]:
df = pd.read_csv("metadata.csv")
# Drop rows with missing values
unique_categories = df["Category"].dropna().unique()

# translation map
category_to_en = {
    cat : translate_danish_to_english(cat)
    for cat in unique_categories
}

df["Category"] = df["Category"].map(category_to_en)

# Save to a new csv
df.to_csv("metadata_en.csv", index=False)

In [72]:
df = pd.read_csv("metadata_en.csv")
# drop where values are NaN
df = df.dropna(subset=["Image File", "Category"])

# use the first 100, can be changed for other subsets
df = df[:25]
true_labels = list(df["Category"])
categories = list(df["Category"].unique())

image_paths = [
    os.path.join("images2", fname) # change images2 to name of folder with images
    for fname in df["Image File"]
]

In [73]:
categories

['Recreation & Garden',
 'Lamps',
 'Electronics',
 'Housing',
 'Music & Books',
 'Missing category',
 'Kitchen stuff',
 'Toys',
 'Bicycles',
 'Furniture']

In [74]:
prompt = f"Given an image, output ONLY ONE Category from this list: {categories}. "
prompt

"Given an image, output ONLY ONE Category from this list: ['Recreation & Garden', 'Lamps', 'Electronics', 'Housing', 'Music & Books', 'Missing category', 'Kitchen stuff', 'Toys', 'Bicycles', 'Furniture']. "

In [83]:
def classify_image(image_path: str, prompt: str):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, text=prompt, return_tensors="pt", padding=True)
    
    generated_ids = model.generate(  
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

    prediction = processor.decode(
        generated_ids[0], skip_special_tokens=True
    )
    return prediction

In [94]:
correct = 0

print(f"{'Predicted':<20} {'Truth':<20} {'Correct?':<10}")
print("-" * 50)
for img_path, gt in zip(image_paths, true_labels):
    pred = classify_image(img_path, prompt)
    is_correct = pred == gt
    if is_correct:
        correct += 1
    print(f"{pred:<20} {gt:<20} {str(is_correct):<10}")

accuracy = correct / len(image_paths)
print("Accuracy:", accuracy)

Predicted            Truth                Correct?  
--------------------------------------------------
Toys                 Recreation & Garden  False     
Lamps                Lamps                True      
Electronics          Electronics          True      
Toys                 Recreation & Garden  False     
Furniture            Housing              False     
Furniture            Housing              False     
Missing category     Recreation & Garden  False     
Music & Books        Music & Books        True      
Lamps                Lamps                True      
Music & Books        Music & Books        True      
Missing category     Recreation & Garden  False     
Furniture            Missing category     False     
Missing category     Housing              False     
Kitchen stuff        Housing              False     
Kitchen stuff        Kitchen stuff        True      
Kitchen stuff        Kitchen stuff        True      
Missing category     Toys                 False 